## Prerequisites - GPU environment check

**Run the cell below first.** It checks the NVIDIA tools (`nvidia-smi`, `nvcc`, `nsys`, `ncu`) and the Python GPU packages (`numpy`, `numba`, `cupy`) this course needs, and prints how to fix anything missing.

- **OK** - ready to use.
- **MISSING** - a required tool; fix it before running this module.
- **optional** - only affects specific bonus paths; the workbooks skip these gracefully.

In [ ]:
# Prerequisites: verify the GPU toolchain before running this notebook.
# This finds check_gpu.py at the repository root and runs it.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

# Ray Tracing and RT Cores Tutorial

This notebook is a practical introduction to ray tracing across modern GPU vendors. It has two goals:

1. make the core ray-tracing algorithm concrete with a small runnable Python example,
2. explain how that maps onto hardware-accelerated ray tracing stacks on NVIDIA, AMD, and Intel.

## Learning goals

- Understand the structure of a basic ray tracer.
- See why brute-force intersection work grows quickly.
- Learn what RT hardware accelerates and what still runs in regular GPU code.
- Compare vendor-specific paths for NVIDIA, AMD, and Intel.
- Know when to choose OptiX, HIP RT, Embree SYCL, or a portable API such as Vulkan Ray Tracing.

## RT hardware in one sentence

Dedicated ray-tracing hardware mainly accelerates:

- BVH traversal,
- ray-box intersection,
- ray-triangle intersection.

It does **not** replace shading, material evaluation, launch logic, buffer management, or scene setup. That is why every vendor still needs a software layer above the hardware.

## CPU baseline: editable reference implementation

The CPU baseline is now stored in a separate editable file so the notebook matches the pattern used for the GPU example:

- `09/edit/cpu_ray_tracer_baseline.py`

That file contains:

- the sphere data structure,
- the ray-sphere intersection helper,
- the baseline image renderer,
- the simple scaling study.

Run the next cell after editing the file. It executes the script in the notebook namespace so later sections can still reuse helpers such as `normalize(...)` and `render_scene(...)`.

In [ ]:
from pathlib import Path

script_candidates = [
    Path('09/edit/cpu_ray_tracer_baseline.py'),
    Path('edit/cpu_ray_tracer_baseline.py'),
]
cpu_script = next((path for path in script_candidates if path.exists()), None)
if cpu_script is None:
    raise FileNotFoundError('Could not find 09/edit/cpu_ray_tracer_baseline.py')

print(f'%run -i {cpu_script.as_posix()}')
get_ipython().run_line_magic('run', f'-i {cpu_script.as_posix()}')

## Physics view: photon propagation from distant galaxies

For physics applications, ray tracing becomes a transport problem rather than a shading problem.

A useful starting model is to treat each photon or photon packet as carrying:

- position,
- direction,
- energy or wavelength,
- statistical weight,
- time or affine parameter,
- optional polarization or species information.

Then the propagation loop becomes:

1. emit a photon packet from a source or trace a line of sight backward from the detector,
2. find the next interaction candidate or boundary crossing,
3. update distance traveled and redshift,
4. apply absorption, scattering, or conversion probabilities,
5. continue until the packet is absorbed, escapes, or reaches the detector.

For light from far galaxies, a first useful approximation is:

- straight-line propagation in a flat background geometry,
- cosmological redshift from the source distance,
- attenuation from intervening gas or dust clouds,
- optional deflections or lensing added later.

This is already enough to prototype many astrophysics workflows:

- line-of-sight attenuation studies,
- toy extragalactic background light models,
- survey forward models,
- Monte Carlo radiative transfer scaffolding,
- later extension toward lensing, plasma effects, or particle transport.

The example below is intentionally simple. It models:

- galaxies on a distant source shell,
- an observer at the origin,
- spherical absorbers in the foreground,
- a toy redshift-distance relation,
- observed flux reduced by geometric dilution and optical depth.

It is not a full cosmology solver, but it is a good starting point for physics application development because the data structures and control flow match what a larger GPU implementation would need.

In [ ]:
from dataclasses import dataclass

@dataclass
class Absorber:
    center: np.ndarray
    radius: float
    density: float
    opacity: float


def segment_sphere_path_length(start, end, sphere):
    direction = end - start
    length = np.linalg.norm(direction)
    if length < 1e-12:
        return 0.0

    direction = direction / length
    oc = start - sphere.center
    b = 2.0 * np.dot(oc, direction)
    c = np.dot(oc, oc) - sphere.radius ** 2
    discriminant = b * b - 4.0 * c
    if discriminant <= 0.0:
        return 0.0

    sqrt_disc = np.sqrt(discriminant)
    t0 = (-b - sqrt_disc) / 2.0
    t1 = (-b + sqrt_disc) / 2.0

    t_enter = max(0.0, min(t0, t1))
    t_exit = min(length, max(t0, t1))
    if t_exit <= t_enter:
        return 0.0
    return t_exit - t_enter


def toy_redshift(distance_mpc, hubble_distance_mpc=4300.0):
    # Toy relation valid only as a simple starting point.
    return distance_mpc / hubble_distance_mpc


def observed_flux(luminosity, distance_mpc, tau):
    z = toy_redshift(distance_mpc)
    d_l = distance_mpc * (1.0 + z)
    return luminosity * np.exp(-tau) / (4.0 * np.pi * d_l * d_l), z


rng = np.random.default_rng(7)
observer = np.array([0.0, 0.0, 0.0])
num_galaxies = 250
num_absorbers = 35
source_shell_mpc = 1200.0

# Random galaxies on a distant shell.
galaxy_dirs = normalize(rng.normal(size=(num_galaxies, 3)))
galaxy_positions = source_shell_mpc * galaxy_dirs
galaxy_luminosities = rng.uniform(0.8, 1.2, size=num_galaxies)

# Random foreground absorbers distributed between source shell and observer.
absorbers = []
for _ in range(num_absorbers):
    radius = rng.uniform(30.0, 90.0)
    center = rng.uniform(-500.0, 500.0, size=3)
    center[2] -= rng.uniform(150.0, 900.0)
    absorbers.append(
        Absorber(
            center=center,
            radius=radius,
            density=rng.uniform(0.005, 0.03),
            opacity=rng.uniform(0.6, 1.4),
        )
    )

fluxes = np.zeros(num_galaxies)
redshifts = np.zeros(num_galaxies)
optical_depths = np.zeros(num_galaxies)

for i, (pos, luminosity) in enumerate(zip(galaxy_positions, galaxy_luminosities)):
    distance = np.linalg.norm(pos - observer)
    tau = 0.0
    for absorber in absorbers:
        path_length = segment_sphere_path_length(observer, pos, absorber)
        tau += absorber.opacity * absorber.density * path_length
    fluxes[i], redshifts[i] = observed_flux(luminosity, distance, tau)
    optical_depths[i] = tau

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sky_x = galaxy_dirs[:, 0] / np.clip(1.0 - galaxy_dirs[:, 2], 1e-6, None)
sky_y = galaxy_dirs[:, 1] / np.clip(1.0 - galaxy_dirs[:, 2], 1e-6, None)

scatter = axes[0].scatter(sky_x, sky_y, c=np.log10(fluxes), s=30, cmap='viridis')
axes[0].set_title('Observed sky map of toy galaxy sample')
axes[0].set_xlabel('projected sky x')
axes[0].set_ylabel('projected sky y')
plt.colorbar(scatter, ax=axes[0], label='log10(observed flux)')

axes[1].scatter(redshifts, fluxes, c=optical_depths, cmap='magma', s=30)
axes[1].set_title('Flux vs redshift with foreground attenuation')
axes[1].set_xlabel('toy redshift z')
axes[1].set_ylabel('observed flux')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f'Galaxies:          {num_galaxies}')
print(f'Foreground clouds: {num_absorbers}')
print(f'Mean optical depth: {optical_depths.mean():.3f}')
print(f'Max optical depth:  {optical_depths.max():.3f}')

## How this maps to a real physics code

The toy example above is deliberately simple, but its architecture is close to what you would scale up:

This is a good development path because the same structure scales to real physics codes:

- replace the toy redshift law with a real cosmology,
- replace simple spheres with density fields, halos, or meshes,
- replace deterministic attenuation with Monte Carlo interaction sampling,
- accelerate intersection queries with OptiX, HIP RT, or Embree SYCL once the CPU model is correct.

### CPU prototype stage

Keep a photon packet structure with fields such as:

- `x, y, z`
- `dx, dy, dz`
- `energy`
- `weight`
- `time`
- `alive`

Add physics modules for:

- cosmology or background geometry,
- absorption and scattering cross sections,
- magnetic or gravitational deflection,
- detector response and selection effects.

### GPU acceleration stage

The most expensive operation is usually finding what a photon ray intersects next.
That is where ray-tracing acceleration structures become useful.

Examples:

- **NVIDIA OptiX**: build acceleration structures over halos, clouds, mesh geometry, or detector elements.
- **AMD HIP RT**: use HIP kernels and HIP RT traversal for the same scene representation.
- **Intel Embree SYCL**: keep the same ray-query logic but run through Embree on Intel GPUs.

### Physics refinements you would add next

1. Replace the toy redshift relation with a proper cosmology model.
2. Replace the toy optical depth with line integrals through density fields.
3. Add wavelength dependence so opacity becomes a function of photon energy.
4. Add stochastic scattering or absorption events with Monte Carlo sampling.
5. Add curved trajectories if the application needs gravitational lensing or plasma effects.

### Key engineering point

For many astrophysics problems, you do not need to start with a full graphics renderer.
You need:

- ray or photon packet transport,
- acceleration structures for geometry or matter regions,
- Monte Carlo interaction sampling,
- a physically meaningful state update per propagation step.

That is why ray-tracing libraries can be useful for physics, even though the end goal is not image synthesis.

## GPU C++ example: photon propagation in CUDA

The Python transport model above is useful for prototyping physics, but a production GPU implementation usually moves the propagation loop into CUDA C++ or another compiled GPU language.

The editable CUDA example for this notebook lives in:

- `09/edit/photon_transport_cuda/photon_propagation_cuda.cu`
- `09/edit/photon_transport_cuda/CMakeLists.txt`

What this example does:

- launches one CUDA thread per source galaxy or detector line of sight,
- stores source positions, luminosities, and absorber clouds in GPU memory,
- computes line-of-sight path length through each spherical absorber,
- accumulates optical depth `tau` on the GPU,
- converts distance to a toy redshift and computes attenuated observed flux.

How to read the CUDA file:

1. `Vec3` and `Absorber` define the minimal data structures for transport.
2. `segmentSpherePathLength(...)` is the geometry query used by each thread.
3. `propagatePhotonsKernel(...)` is the core GPU loop: one thread processes one source.
4. `buildSourceShell(...)` and `buildAbsorbers(...)` generate a deterministic toy dataset on the host.
5. `main()` allocates unified memory, launches the kernel, and prints summary statistics.

The source file itself is annotated with comments at the places where the physics model maps onto CUDA execution.

In [ ]:
# This cell prints the commands needed to build and run the editable CUDA C++ photon transport example.
# It does not compile the code here because the current environment does not expose nvcc.

from pathlib import Path

photon_project = Path('09/edit/photon_transport_cuda')
if not photon_project.exists():
    photon_project = Path('edit/photon_transport_cuda')
if not photon_project.exists():
    raise FileNotFoundError('Could not find 09/edit/photon_transport_cuda')

print('Editable CUDA photon propagation files:')
for path in sorted(photon_project.iterdir()):
    print(f'  - {path.as_posix()}')

print('\nRecommended Windows nvcc commands:')
print(f'cd {photon_project.as_posix()}')
print(r'nvcc -O2 -std=c++17 photon_propagation_cuda.cu -o photon_propagation_cuda.exe')
print(r'.\photon_propagation_cuda.exe')

print('\nOptional CMake flow:')
print(r'cmake -S . -B build')
print(r'cmake --build build --config Release')
print(r'.\build\Release\photon_propagation_cuda.exe')

## What RT hardware accelerates

A useful mental model is:

- regular shader or compute cores handle programmable shading and control flow,
- RT hardware handles BVH traversal and common ray-geometry intersection work,
- tensor or matrix engines are often used around denoising or learned rendering workflows.

In a real renderer, the flow usually looks like this:

1. build or update acceleration structures,
2. launch rays,
3. traverse the BVH,
4. intersect against candidate geometry,
5. run miss and hit programs,
6. shade, accumulate, and optionally denoise.

This is why RT hardware speeds up ray tracing without removing the need for a full rendering API or framework.

The next section shows the first NVIDIA-style GPU step before OptiX: a plain CUDA kernel where one thread traces one pixel. The editable source lives in `09/edit/gpu_ray_tracer_numba.py`, and the notebook runs that file directly.

## GPU example: editable NVIDIA-style CUDA ray tracer

To keep the notebook readable and make the implementation easy to modify, the runnable GPU example is stored in a separate file:

- `09/edit/gpu_ray_tracer_numba.py`

This example shows:

- a 2D CUDA launch over the image,
- one thread per pixel,
- sphere data copied into device arrays,
- manual ray-sphere intersection in a CUDA kernel,
- validation against a CPU reference image.

Run the next cell after editing the file. If CUDA is visible, it uses the real NVIDIA device. If CUDA is not visible, it falls back to the Numba CUDA simulator so the example still executes in this notebook environment.

In [ ]:
from pathlib import Path

script_candidates = [
    Path('09/edit/gpu_ray_tracer_numba.py'),
    Path('edit/gpu_ray_tracer_numba.py'),
]
gpu_script = next((path for path in script_candidates if path.exists()), None)
if gpu_script is None:
    raise FileNotFoundError('Could not find 09/edit/gpu_ray_tracer_numba.py')

print(f'%run -i {gpu_script.as_posix()}')
get_ipython().run_line_magic('run', f'-i {gpu_script.as_posix()}')

In [ ]:
from pathlib import Path

optix_project = Path('09/edit/optix_minimal')
if not optix_project.exists():
    optix_project = Path('edit/optix_minimal')
if not optix_project.exists():
    raise FileNotFoundError('Could not find 09/edit/optix_minimal')

print('Editable OptiX example files:')
for path in sorted(optix_project.iterdir()):
    print(f'  - {path.as_posix()}')

print('\nRecommended Windows commands on a machine with CUDA Toolkit, OptiX SDK, and Visual Studio:')
print(f'cd {optix_project.as_posix()}')
print(r'cmake -S . -B build -DOptiX_INSTALL_DIR="C:\path\to\NVIDIA-OptiX-SDK"')
print(r'cmake --build build --config Release')
print(r'.\build\Release\minimal_optix7_outline.exe')

## Next step after plain CUDA: editable minimal OptiX project

The plain CUDA renderer above is useful because it makes the per-pixel execution model explicit. The next NVIDIA-specific step is to keep ray generation logic but move intersection acceleration into OptiX.

The editable OptiX example files live here:

- `09/edit/optix_minimal/minimal_optix7_outline.cu`
- `09/edit/optix_minimal/CMakeLists.txt`

This small project is intentionally only a host-side skeleton. It shows the order in which an OptiX application starts up, creates the device context, and prepares for a later `optixLaunch(...)`. It is not executed in this notebook because the current environment does not expose an OptiX SDK toolchain.

## Vendor comparison: NVIDIA, AMD, Intel

### NVIDIA: OptiX
Best fit if you want direct access to NVIDIA RTX hardware from a compute-style application.

Use it when:

- you are on NVIDIA GPUs only,
- you want explicit control over launch parameters, program groups, and the Shader Binding Table,
- you are already comfortable with CUDA-style host and device programming.

Typical software stack:

- CUDA for memory and execution setup,
- OptiX for modules, program groups, acceleration structures, SBT, and ray launch,
- optional OptiX denoiser for post-processing.

### AMD: HIP RT
HIP RT is AMD's low-level ray tracing library for HIP. It is designed to integrate into HIP applications without inventing a completely separate shader model.

Use it when:

- you want a low-level vendor path on AMD GPUs,
- your renderer or compute code is already in HIP,
- you want hardware-accelerated ray tracing on RDNA 2 or newer AMD GPUs.

Key ideas:

- build geometry or scene acceleration structures through HIP RT,
- launch ordinary HIP kernels,
- use traversal objects such as closest-hit or any-hit traversal inside HIP device code.

Important nuance:

- HIP RT can load HIP and CUDA APIs dynamically and can run on AMD and NVIDIA GPUs,
- hardware-accelerated RT in HIP RT is the AMD path and requires RDNA 2 or newer.

### Intel: Embree with SYCL
For Intel GPUs, the closest low-level comparison is Embree's SYCL API rather than a separate OptiX-like proprietary API.

Use it when:

- you are targeting Intel GPUs and are comfortable with SYCL / oneAPI DPC++,
- you want a ray tracing API that is close to Embree's established CPU interface,
- you want one renderer structure that can target Intel CPU and Intel GPU backends.

Key ideas:

- create an Embree SYCL device from a SYCL context,
- build scenes and geometry similarly to normal Embree,
- submit SYCL kernels and trace rays with the Embree SYCL API on supported Intel GPUs.

Important nuance:

- Embree is historically famous as a CPU ray tracing library,
- current Embree also supports hardware-accelerated ray tracing on Intel GPUs through SYCL,
- the GPU path is specifically for supported Intel Xe HPG and Xe HPC class devices.

### Portable alternative
If you want one graphics-oriented path across vendors, use Vulkan Ray Tracing or DXR instead of vendor libraries.

### Rule of thumb

- Learn **OptiX** if the goal is NVIDIA-specific RT workflows.
- Learn **HIP RT** if the goal is AMD-specific RT workflows in HIP.
- Learn **Embree SYCL** if the goal is Intel CPU plus Intel GPU rendering with a similar API shape.
- Use **Vulkan RT** or **DXR** if the goal is portability across GPU vendors.

## Side-by-side API shape

The three vendor paths can be compared at the same conceptual level.

### NVIDIA OptiX
```cpp
initCuda();
initOptix();
buildOptixModules();
createProgramGroups();
linkPipeline();
buildAccelerationStructures();
buildShaderBindingTable();
optixLaunch(...);
```

### AMD HIP RT
```cpp
hipInit(...);
hiprtCreateContext(...);
hiprtCreateGeometry(...);
hiprtCreateScene(...);
buildHiprtAccelerationStructures(...);
launchHipKernel<<<...>>>(scene, ...);

// inside HIP device code
hiprtSceneTraversalClosest tr(scene, ray);
hiprtHit hit = tr.getNextHit();
```

### Intel Embree SYCL
```cpp
sycl::device gpu(rtcSYCLDeviceSelector);
sycl::queue queue(gpu);
RTCDevice device = rtcNewSYCLDevice(queue.get_context(), "");
RTCScene scene = rtcNewScene(device);
attachGeometry(scene, ...);
rtcCommitScene(scene);

queue.parallel_for(..., [=]() {
    // trace ray against the committed scene
    rtcIntersect1(...);
});
```

## How to read this comparison

At a high level all three do the same job:

- initialize a device or context,
- build geometry and acceleration structures,
- define rays,
- issue closest-hit or any-hit style queries,
- use the result for shading.

The main difference is where the abstraction boundary sits:

- **OptiX** has the richest dedicated ray tracing pipeline abstraction.
- **HIP RT** stays close to HIP kernels and device-side traversal objects.
- **Embree SYCL** stays close to Embree's ray query model, but uses SYCL to run on Intel GPUs.

## Minimal OptiX 7 anatomy

A minimal OptiX application usually does the following:

- create a CUDA context or stream,
- create an OptiX device context,
- compile device programs for ray generation, miss, and hit stages,
- create program groups,
- link them into an OptiX pipeline,
- build a geometry acceleration structure (GAS) and sometimes an instance acceleration structure (IAS),
- populate the Shader Binding Table (SBT),
- call `optixLaunch(...)`.

The editable source for the host-side startup skeleton now lives in:

- `09/edit/optix_minimal/minimal_optix7_outline.cu`
- `09/edit/optix_minimal/CMakeLists.txt`

That is the conceptual bridge from the toy ray tracer in this notebook to a real RTX renderer.

## Windows setup notes for a first OptiX run

A typical Windows setup is:

- NVIDIA RTX GPU with a recent driver,
- CUDA Toolkit,
- OptiX SDK,
- CMake 3.17 or newer,
- Visual Studio 2022 with C++ tools.

If the OptiX SDK is not installed in its default location, set the version-specific environment variable expected by the sample repository before configuring CMake.

A typical command-line flow is:

```powershell
git clone https://github.com/NVIDIA/OptiX_Apps.git
cd OptiX_Apps
cmake -S . -B build
cmake --build build --config Release
```

Then run one of the introductory executables from the build output directory.

Useful progression:

1. `intro_runtime`
2. `intro_driver`
3. `intro_denoiser`
4. `intro_motion_blur`
5. `rtigo9_omm`

## Suggested exercises

1. Add shadows to the Python baseline by shooting one shadow ray per visible point.
2. Add a reflective sphere and watch the ray count grow.
3. Increase the number of primitives and measure how the brute-force baseline scales.
4. Read an OptiX sample and identify where it creates the device context, modules, program groups, pipeline, SBT, and acceleration structure.
5. Compare OptiX with Vulkan RT or DXR only after the OptiX flow is clear.